In [ ]:
import re
import json
from pathlib import Path

def extract_not_blocks(text):
    not_blocks = []
    i = 0
    n = len(text)

    while i < n:
        if text[i:i+3].upper() == 'NOT':
            j = i + 3
            while j < n and text[j].isspace():
                j += 1

            if j < n and text[j] == '(':
                start = j
                stack = []
                k = j

                while k < n:
                    if text[k] == '(':
                        stack.append('(')
                    elif text[k] == ')':
                        if not stack:
                            break
                        stack.pop()
                        if not stack:
                            end = k
                            not_blocks.append(text[start:end+1])
                            i = k
                            break
                    k += 1

                i += 1
                continue

        i += 1

    return not_blocks


def remove_blocks_from_text(text, blocks):
    cleaned = text
    for b in blocks:
        cleaned = cleaned.replace(b, " ")
    return cleaned


def extract_patterns(text):
    pattern = r'(TITLE-ABS-KEY|TITLE-ABS|AUTHKEY)\("([^"]+)"\)'
    matches = re.findall(pattern, text, flags=re.IGNORECASE)

    results = {"TITLE_ABS_KEY": [], "TITLE_ABS": [], "AUTHKEY": []}

    for field, phrase in matches:
        f = field.upper()
        phrase_clean = phrase.strip()

        if f == "TITLE-ABS-KEY":
            results["TITLE_ABS_KEY"].append(phrase_clean)
        elif f == "TITLE-ABS":
            results["TITLE_ABS"].append(phrase_clean)
        else:
            results["AUTHKEY"].append(phrase_clean)

    return results


def parse_elsevier_rule_final(file_path):
    raw = Path(file_path).read_text(encoding="utf-8")
    text = " ".join(raw.split())

    not_blocks = extract_not_blocks(text)
    exclude_text = " ".join(not_blocks)
    exclude_patterns = extract_patterns(exclude_text)

    cleaned_text = remove_blocks_from_text(text, not_blocks)
    include_patterns = extract_patterns(cleaned_text)

    # Deduplication
    def dedup(seq):
        seen = set()
        out = []
        for s in seq:
            if s not in seen:
                out.append(s)
                seen.add(s)
        return out

    parsed = {
        "include": {
            "TITLE_ABS": dedup(include_patterns["TITLE_ABS"]),
            "AUTHKEY": dedup(include_patterns["AUTHKEY"]),
            "TITLE_ABS_KEY": dedup(include_patterns["TITLE_ABS_KEY"])
        },
        "exclude": {
            "TITLE_ABS": dedup(exclude_patterns["TITLE_ABS"]),
            "AUTHKEY": dedup(exclude_patterns["AUTHKEY"]),
            "TITLE_ABS_KEY": dedup(exclude_patterns["TITLE_ABS_KEY"])
        }
    }

    return parsed


def save_parsed_rule_to_json(parsed_rule, output_path):
    Path(output_path).write_text(
        json.dumps(parsed_rule, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )


def print_rule_summary(parsed):
    print("\n===== SUMMARY =====")

    for section in ["include", "exclude"]:
        print(f"\n[{section.upper()}]")
        total = sum(len(parsed[section][k]) for k in parsed[section])
        print(f"Total: {total}")

        for cat in ["TITLE_ABS", "AUTHKEY", "TITLE_ABS_KEY"]:
            print(f"  {cat}: {len(parsed[section][cat])}")

In [ ]:
# ==========================
# PROCESS ALL SDG FILES 1–17
# ==========================

def process_all_sdg_rules():
    for i in range(1, 18):
        sdg_txt = f"/content/SDG{str(i).zfill(2)}.txt"
        sdg_json = f"SDG{str(i).zfill(2)}.json"

        # cek apakah file txt ada
        if not Path(sgd_txt := sdg_txt).exists():
            print(f"[SKIP] {sdg_txt} tidak ditemukan.")
            continue

        print(f"\n===================================")
        print(f"Processing {sdg_txt}")
        print(f"===================================")

        # parse
        parsed = parse_elsevier_rule_final(sdg_txt)

        # print summary
        print_rule_summary(parsed)

        # save json
        save_parsed_rule_to_json(parsed, sdg_json)
        print(f"Saved → {sdg_json}")


# --- RUN SEKALI UNTUK SDG01–SDG17 ---
process_all_sdg_rules()



Processing /content/SDG01.txt

===== SUMMARY =====

[INCLUDE]
Total: 744
  TITLE_ABS: 356
  AUTHKEY: 388
  TITLE_ABS_KEY: 0

[EXCLUDE]
Total: 135
  TITLE_ABS: 45
  AUTHKEY: 45
  TITLE_ABS_KEY: 45
Saved → SDG01.json

Processing /content/SDG02.txt

===== SUMMARY =====

[INCLUDE]
Total: 2013
  TITLE_ABS: 931
  AUTHKEY: 928
  TITLE_ABS_KEY: 154

[EXCLUDE]
Total: 99
  TITLE_ABS: 49
  AUTHKEY: 49
  TITLE_ABS_KEY: 1
Saved → SDG02.json

Processing /content/SDG03.txt

===== SUMMARY =====

[INCLUDE]
Total: 4588
  TITLE_ABS: 2294
  AUTHKEY: 2294
  TITLE_ABS_KEY: 0

[EXCLUDE]
Total: 73
  TITLE_ABS: 36
  AUTHKEY: 37
  TITLE_ABS_KEY: 0
Saved → SDG03.json

Processing /content/SDG04.txt

===== SUMMARY =====

[INCLUDE]
Total: 1481
  TITLE_ABS: 739
  AUTHKEY: 742
  TITLE_ABS_KEY: 0

[EXCLUDE]
Total: 89
  TITLE_ABS: 19
  AUTHKEY: 19
  TITLE_ABS_KEY: 51
Saved → SDG04.json

Processing /content/SDG05.txt

===== SUMMARY =====

[INCLUDE]
Total: 1221
  TITLE_ABS: 610
  AUTHKEY: 611
  TITLE_ABS_KEY: 0

[EXCLUD